# Tests rapides des LLMs

Comparaison a la main des 3 modeles sur quelques questions pour se faire une idee avant de lancer le vrai benchmark de `02_experiments`.

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline

In [ ]:
# on charge embedder + chromadb une seule fois
emb = Embedder('OrdalieTech/Solon-embeddings-base-0.1')
vs = VectorStore('../data/chroma_db', 'collection_solon_base')
print('chunks:', vs.count())

In [ ]:
# quelques questions tests (une par type de source pour varier)
questions = [
    "Dois-je nommer un DPO si j'ai 30 salaries ?",
    "Combien de temps puis-je conserver un CV de candidat non retenu ?",
    "Pourquoi Google a ete sanctionne en 2019 ?",
]

## Boucle sur les 3 modeles

Du plus petit au plus gros. On libere la memoire entre chaque pour pas saturer.

In [ ]:
def free_mem():
    gc.collect()
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

modeles = [
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'Qwen/Qwen2.5-1.5B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

llm = None
for m in modeles:
    print('=' * 60)
    print('modele:', m)
    if llm is not None:
        del llm
    free_mem()
    llm = LLMGenerator(m)
    pipe = RAGPipeline(emb, vs, llm, prompt_name='citation')
    for q in questions:
        print()
        print('Q:', q)
        resp = pipe.answer(q)
        print('R:', resp.answer[:600])
        print('  latence total:', resp.latency_ms['total'], 'ms')

## Observations

A completer a la main apres avoir regarde les reponses:
- TinyLlama: ...
- Qwen 1.5B: ...
- Qwen 3B: ...